In [34]:
import json
with open("../chunk_result/upsert_data_200d.json","r",encoding="utf-8") as f:
    r6_upsert = json.load(f)

In [35]:
r6_upsert[50]['id']

'chunk-200d-50'

In [ ]:
import re
# 변환 적용 # r6 만 적용
for item in r6_upsert:
    if "id" in item and isinstance(item["id"], str):  # id 값이 문자열인지 확인
        item["id"] = re.sub(r'chunk-(\d+)', r'chunk-r6-\1', item["id"])

In [36]:
r6_upsert[50]['metadata']['pdf_path']

'/Users/yoeun/Library/Mobile Documents/com~apple~CloudDocs/github/FINAL Project/parse&chunk/data/pdf/split_pdf_image/200d/200d_page_62.jpg'

In [37]:
import re

# 변환 적용
for item in r6_upsert:
    if "metadata" in item and "image_path" in item["metadata"]:
        # 기존 경로 리스트 가져오기
        image_paths = item["metadata"]["image_path"]

        # 새로운 경로 리스트 생성
        updated_paths = []
        for path in image_paths:
            # 정규식으로 기존 경로에서 마지막 파일명만 추출
            match = re.search(r'200d_page_\d+_\d+\.png$', path)
            if match:
                filename = match.group()  # 파일명만 추출
                new_path = f"/canon/200d/{filename}"  # 새로운 경로 생성
                updated_paths.append(new_path)
            else:
                updated_paths.append(path)  # 매칭 안 되면 원본 유지

        # 업데이트된 경로 리스트 적용
        item["metadata"]["image_path"] = updated_paths

    if "metadata" in item and "pdf_path" in item["metadata"]:
        # 기존 경로 리스트 가져오기
        image_paths = item["metadata"]["pdf_path"]

        # 새로운 경로 리스트 생성
        match = re.search(r'200d_page_\d+\.jpg$', image_paths)
          
        if match:
            filename = match.group()  # 파일명만 추출
            new_path = f"/canon/pdf/200d/{filename}"  # 새로운 경로 생성

        # 업데이트된 경로 리스트 적용
        item["metadata"]["pdf_path"] = new_path

In [38]:
r6_upsert[44]['metadata']['image_path']

['/canon/200d/200d_page_56_1.png',
 '/canon/200d/200d_page_56_2.png',
 '/canon/200d/200d_page_56_3.png']

In [39]:
r6_upsert[44]['metadata']['pdf_path']

'/canon/pdf/200d/200d_page_56.jpg'

In [40]:
# upsert data save
with open("../data/upsert_img/upsert_data_200d_img.json", "w", encoding='utf-8') as f:
    json.dump(r6_upsert, f, ensure_ascii=False, indent=4)

In [12]:
from pinecone import Pinecone
from pinecone import ServerlessSpec
import os

pinecone_api = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api)
index_name = "canonmodel"

index = pc.Index(index_name)

In [13]:
import numpy as np

BATCH_SIZE = 100  # 한 번에 업로드할 벡터 수 (조절 가능)

# 데이터를 배치 단위로 업로드하는 함수
def upsert_in_batches(index, data, batch_size=BATCH_SIZE):
    for i in range(0, len(data), batch_size):
        batch = data[i : i + batch_size]
        index.upsert(vectors=batch)
        print(f"Upserted {len(batch)} vectors.")

In [ ]:
r6_upsert[0]

{'id': 'chunk-r50-0',
 'values': [0.06825494021177292,
  0.04057204723358154,
  -0.03851243853569031,
  0.0070923566818237305,
  -0.03880034014582634,
  -0.04987349733710289,
  -0.004916481673717499,
  -0.008288257755339146,
  0.011394278146326542,
  -0.021415485069155693,
  0.0393761470913887,
  -0.007861941121518612,
  -0.02537967450916767,
  -0.018935097381472588,
  0.01813783124089241,
  -0.030761228874325752,
  -0.06847640126943588,
  0.02969820611178875,
  0.02819225750863552,
  -0.020562851801514626,
  0.03957546129822731,
  0.019599488005042076,
  0.007546356413513422,
  -0.029742499813437462,
  -0.0383131206035614,
  -0.04280882328748703,
  -0.012634471990168095,
  -0.01658758893609047,
  0.021703386679291725,
  0.012114033102989197,
  -0.005353871267288923,
  -0.03120415471494198,
  -0.010796328075230122,
  -0.05935211852192879,
  -0.04072707146406174,
  0.03220073878765106,
  0.03844600170850754,
  -0.0275943074375391,
  -0.006760162301361561,
  -0.006333845667541027,
  -0.0

In [41]:
# Pinecone 업로드 (배치 업로드 실행)
upsert_in_batches(index, r6_upsert)
print(f"Upserted {len(r6_upsert)} chunks into Pinecone.")

Upserted 100 vectors.
Upserted 100 vectors.
Upserted 100 vectors.
Upserted 100 vectors.
Upserted 94 vectors.
Upserted 494 chunks into Pinecone.
